# 02 — Building a target and shooting rays through it

*Weeks 3–4*

In this notebook you build the virtual target and implement the first student-owned analysis function. The ray-marching physics is supplied in `muography.py`; your job is to assemble it into an opacity map.


In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive.*')
sys.path.insert(0, os.path.abspath('../code'))
import numpy as np
import matplotlib.pyplot as plt
import muography as mg
import project_config as cfg
import student_analysis as sa
print('ready')


ready


## The target

`mg.Pyramid` is a square-based pyramid with a 230 m base and 139 m height. The baseline chamber is a **10 m radius (20 m diameter)** sphere centred at `(0, 0, 70)`. These values are stored in `project_config.py` so that all notebooks use the same baseline.


In [2]:
target = mg.Pyramid(
    base=cfg.PYRAMID_BASE_M, height=cfg.PYRAMID_HEIGHT_M,
    density=cfg.ROCK_DENSITY_G_CM3, void_centre=cfg.VOID_CENTRE_M,
    void_radius=cfg.VOID_RADIUS_M, has_void=True,
)
print('inside the void  :', target.density_at(0, 0, 70), 'g/cm^3')
print('inside the rock  :', target.density_at(0, 0, 30), 'g/cm^3')
print('outside entirely :', target.density_at(200, 0, 10), 'g/cm^3')


inside the void  : 0.0012 g/cm^3
inside the rock  : 2.5 g/cm^3
outside entirely : 0.0012 g/cm^3


### YOUR TURN — look at what you have built

Make a vertical slice through the middle of the pyramid and show the chamber. Label the axes in metres and the colourbar in `g/cm³`.


In [3]:
# YOUR TURN
y_vals = np.linspace(-150, 150, 300)
z_vals = np.linspace(0, 150, 200)
Y, Z = np.meshgrid(y_vals, z_vals, indexing='xy')

rho = np.zeros_like(Y)
for i in range(Y.shape[0]):
    for j in range(Y.shape[1]):
        rho[i, j] = target.density_at(0.0, Y[i, j], Z[i, j])

plt.pcolormesh(Y, Z, rho, shading='auto', cmap='gray_r')
plt.xlabel('y (m)')
plt.ylabel('z (m)')
plt.title('Vertical slice through the pyramid at x = 0')
plt.colorbar(label=r'density (g/cm$^3$)')
plt.gca().set_aspect('equal')
plt.savefig('../figures/05_density_slice.png', dpi=300, bbox_inches='tight')
plt.show()

## Where does the detector go?

The baseline detector position is `(0, -40, 3)` m. The direction functions use two pointing angles, `ax` and `ay`; together they define the viewing direction.


In [4]:
detector = np.array(cfg.DETECTOR_POSITION_M, dtype=float)
d = mg.direction_from_angles(0.0, 0.0)
print('straight up:', d, '-> zenith', np.degrees(mg.zenith_from_direction(d)), 'deg')


straight up: [0. 0. 1.] -> zenith 0.0 deg


## Ray marching — supplied physics

`mg.trace_one_ray` is the supplied single-ray function. Read it before using it. It walks along a ray in 0.25 m steps and accumulates density × path length.


In [5]:
for deg in [0, 10, 20, 25, 30, 35, 40]:
    d = mg.direction_from_angles(0.0, np.radians(deg))
    X = mg.trace_one_ray(target, detector, d, step=cfg.RAY_STEP_M)
    print(f'ay = {deg:3d} deg   opacity = {X:9.0f} g/cm^2   ({X/100/2.5:6.1f} m of limestone)')


ay =   0 deg   opacity =     21999 g/cm^2   (  88.0 m of limestone)
ay =  10 deg   opacity =     28371 g/cm^2   ( 113.5 m of limestone)
ay =  20 deg   opacity =     34118 g/cm^2   ( 136.5 m of limestone)
ay =  25 deg   opacity =     29620 g/cm^2   ( 118.5 m of limestone)
ay =  30 deg   opacity =     26497 g/cm^2   ( 106.0 m of limestone)
ay =  35 deg   opacity =     26434 g/cm^2   ( 105.7 m of limestone)
ay =  40 deg   opacity =     29933 g/cm^2   ( 119.7 m of limestone)


### YOUR TURN — find the void by hand

Scan `ay` from 15° to 45° in 1° steps. Plot the opacity for the target and for `target.without_void()`. The chamber should create a notch that is absent in the no-void control.


In [6]:
# YOUR TURN
angles = np.arange(15, 46, 1)
opacity_void = []
opacity_solid = []
for deg in angles:
    d = mg.direction_from_angles(0.0, np.radians(deg))
    opacity_void.append(mg.trace_one_ray(target, detector, d, step=cfg.RAY_STEP_M))
    opacity_solid.append(mg.trace_one_ray(target.without_void(), detector, d, step=cfg.RAY_STEP_M))

# Use a fresh figure: the previous density-slice cell leaves its axes and
# colorbar as the current Matplotlib figure under some backends.
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(angles, opacity_void, label='with void')
ax.plot(angles, opacity_solid, '--', label='no void')
ax.set_xlabel('ay (degrees)')
ax.set_ylabel('Opacity (g/cm²)')
ax.set_title('Chamber notch in opacity')
ax.legend(frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('../figures/06_opacity_notch.png', dpi=300, bbox_inches='tight')
plt.show()


## Making a whole image

### YOUR TURN — implement the shared function

Open `code/student_analysis.py` and implement:

```python
opacity_map(target, detector, field_deg=40, step_deg=2)
```

It should build `ax` and `ay` grids, trace one ray per direction using `mg.direction_from_angles` and `mg.trace_one_ray`, and return `(opacity, ax_deg, ay_deg)`. Keep the angle grids in degrees for plotting.

**Important:** this function lives in `student_analysis.py`, not only in this notebook. That makes notebooks 03–05 reproducible from a fresh kernel.


In [7]:
# After implementing the function in code/student_analysis.py, run this cell.
opacity, ax_deg, ay_deg = sa.opacity_map(target, detector)
print('shape:', opacity.shape)


shape: (41, 41)


### YOUR TURN — display and validate

Display the opacity map with `origin='lower'`. Then repeat it for `target.without_void()`. Finally, repeat the map with `ray_step=0.125` m and compare it with the 0.25 m result. The change should be small.


In [8]:
# YOUR TURN
opacity, ax_deg, ay_deg = sa.opacity_map(target, detector)
opacity_no_void, _, _ = sa.opacity_map(target.without_void(), detector)
opacity_fine, _, _ = sa.opacity_map(target, detector, ray_step=0.125)

extent = [ax_deg[0], ax_deg[-1], ay_deg[0], ay_deg[-1]]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

im0 = axes[0, 0].imshow(opacity, origin='lower', extent=extent, aspect='auto',
                        cmap='viridis')
axes[0, 0].set_title('Opacity with the chamber')
axes[0, 0].set_xlabel('ax (degrees)')
axes[0, 0].set_ylabel('ay (degrees)')
fig.colorbar(im0, ax=axes[0, 0], label=r'opacity (g cm$^{-2}$)')

im1 = axes[0, 1].imshow(opacity_no_void, origin='lower', extent=extent,
                        aspect='auto', cmap='viridis')
axes[0, 1].set_title('Opacity with the chamber filled solid')
axes[0, 1].set_xlabel('ax (degrees)')
axes[0, 1].set_ylabel('ay (degrees)')
fig.colorbar(im1, ax=axes[0, 1], label=r'opacity (g cm$^{-2}$)')

difference = opacity_no_void - opacity
im2 = axes[1, 0].imshow(difference, origin='lower', extent=extent,
                        aspect='auto', cmap='magma')
axes[1, 0].set_title('Chamber signature: solid minus void')
axes[1, 0].set_xlabel('ax (degrees)')
axes[1, 0].set_ylabel('ay (degrees)')
fig.colorbar(im2, ax=axes[1, 0], label=r'$\Delta$opacity (g cm$^{-2}$)')

profile = opacity[:, np.argmin(np.abs(ax_deg))]
profile_fine = opacity_fine[:, np.argmin(np.abs(ax_deg))]
axes[1, 1].plot(ay_deg, profile, 'o-', ms=3, label='ray step 0.25 m')
axes[1, 1].plot(ay_deg, profile_fine, '--', label='ray step 0.125 m')
axes[1, 1].set_xlabel('ay (degrees)')
axes[1, 1].set_ylabel(r'opacity (g cm$^{-2}$)')
axes[1, 1].set_title('Vertical profile at ax = 0')
axes[1, 1].legend(frameon=False)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/07_opacity_maps.png', dpi=300, bbox_inches='tight')
plt.show()

# --- validation numbers ---------------------------------------------------
print('map shape                :', opacity.shape)
print('opacity range            :', f'{opacity.min():.1f} to {opacity.max():.1f} g/cm^2')
print('central opacity          :', f'{opacity[np.argmin(np.abs(ay_deg)), np.argmin(np.abs(ax_deg))]:.1f} g/cm^2')
print('maximum chamber signature:', f'{difference.max():.1f} g/cm^2')
print('hand estimate for 2r     :', f'{2*cfg.VOID_RADIUS_M*cfg.ROCK_DENSITY_G_CM3*100:.1f} g/cm^2')
rel = np.abs(opacity_fine - opacity) / np.maximum(opacity, 1.0)
print('step 0.25 -> 0.125 m     : max relative change',
      f'{100*rel.max():.3f}%, mean {100*rel.mean():.3f}%')
print('no-void control: max |difference| far from chamber (ay < 10 deg):',
      f'{np.max(np.abs(difference[ay_deg < 10])):.2f} g/cm^2')


map shape                : (41, 41)
opacity range            : 14315.2 to 34992.7 g/cm^2
central opacity          : 21999.0 g/cm^2
maximum chamber signature: 4935.1 g/cm^2
hand estimate for 2r     : 5000.0 g/cm^2
step 0.25 -> 0.125 m     : max relative change 0.236%, mean 0.073%
no-void control: max |difference| far from chamber (ay < 10 deg): 0.00 g/cm^2


### What the maps show

The first two panels are almost identical: the pyramid's shape dominates the
opacity. The third panel isolates the chamber by subtracting the twin target.
It is not a plotting trick: this difference is exactly the quantity that the
ratio map in notebook 03 will recover from expected counts.

The chamber signature peaks at about **4,900 g cm$^{-2}$**, which is the opacity
of the rock missing along the longest chord through a 10 m radius sphere:

$$
\Delta\varrho = 2r\rho \times 100
              = 2(10\ \mathrm{m})(2.5\ \mathrm{g\,cm^{-3}})(100)
              \approx 5{,}000\ \mathrm{g\,cm^{-2}}.
$$

The ray does not pass exactly through the centre, so the measured value is a
little smaller than the ideal 5,000. That agreement is the first validation of
the ray tracer.

Halving the ray step from 0.25 m to 0.125 m changes the map by only a fraction
of a percent. The step is therefore fine enough for this geometry; the
remaining difference is a numerical-integration residual, not physics.

The no-void control is flat to within a few g cm$^{-2}$ away from the chamber,
which shows that the difference map is not hiding a geometry bug. Near the
chamber the control is exactly the no-void target, so any nonzero difference
there is the chamber.
